In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

job_corpus = pd.read_csv("../data/processed/job_corpus_clean.csv")
job_corpus = job_corpus.dropna(subset=["clean_text"]).reset_index(drop=True)

print("Job corpus shape:", job_corpus.shape)

# Fit a fresh TF-IDF vectorizer on the job corpus (separate from the resume classifier's vectorizer)
job_tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
job_vectors = job_tfidf.fit_transform(job_corpus["clean_text"])

print("Job vectors shape:", job_vectors.shape)

Job corpus shape: (61019, 8)
Job vectors shape: (61019, 10000)


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text, top_n=10):
    # Clean the resume text the same way the job corpus was cleaned
    cleaned = clean_text(resume_text)
    
    # Transform using the SAME fitted vectorizer (not fit again!)
    resume_vector = job_tfidf.transform([cleaned])
    
    # Compute cosine similarity between this resume and every job
    similarities = cosine_similarity(resume_vector, job_vectors).flatten()
    
    # Get indices of top-N most similar jobs
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = job_corpus.iloc[top_indices][["title", "company", "location", "source"]].copy()
    results["match_score"] = similarities[top_indices]
    return results.reset_index(drop=True)

# We need clean_text function again since this is a new notebook
import re
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [3]:
resumes = pd.read_csv("../data/processed/resumes_clean.csv")
sample_resume = resumes["Resume"].iloc[0]

print("Testing recommender on this resume's category:", resumes["Category"].iloc[0])
print("\nTop 10 recommended jobs:")
recommend_jobs(sample_resume, top_n=10)

Testing recommender on this resume's category: HR

Top 10 recommended jobs:


,title,company,location,source,match_score
0,Service Engineer,Building Control Solutions,Permanent Remote,naukri,0.283172
1,Lower Divisional Clerk,NaN,NaN,linkedin,0.213271
2,Amazon is Hiring For International Voice - Tam...,Amazon,Permanent Remote,naukri,0.200007
3,Area Sales Manager - MNC FMCG CO - Open Tamil ...,NaN,NaN,linkedin,0.197799
4,"Lead Representative, Client Processing",BNY Mellon,Chennai,naukri,0.190334
5,HR Lead / Head / Business Partner (shared Serv...,NaN,NaN,linkedin,0.185676
6,Customer Service Representative,"FPS CHENNAI, INDIA",Pune,naukri,0.185489
7,Associate Research Scientist,NaN,NaN,linkedin,0.184064
8,"Manager- Finance, Administration & HR",NaN,NaN,linkedin,0.182822
9,Human Resource,NaN,NaN,linkedin,0.182577


In [4]:
def precision_at_k(k=10, n_samples=50):
    """
    For a sample of resumes, check what fraction of top-K recommended jobs
    have a title containing a word that overlaps with the resume's known category.
    This is a proxy since we don't have ground-truth job-resume pairs.
    """
    sample = resumes.sample(n=n_samples, random_state=42)
    hits = 0
    total = 0
    
    for _, row in sample.iterrows():
        category_words = row["Category"].lower().replace("-", " ").split()
        recs = recommend_jobs(row["Resume"], top_n=k)
        
        for title in recs["title"]:
            total += 1
            title_lower = str(title).lower()
            if any(word in title_lower for word in category_words):
                hits += 1
    
    return hits / total if total > 0 else 0

score = precision_at_k(k=10, n_samples=50)
print(f"Precision@10 (category-word proxy): {round(score, 4)}")

Precision@10 (category-word proxy): 0.254


In [5]:
categories_to_check = ["CHEF", "ACCOUNTANT", "AVIATION", "DESIGNER"]

for cat in categories_to_check:
    sample_row = resumes[resumes["Category"] == cat].iloc[0]
    print(f"\n{'='*50}")
    print(f"Resume Category: {cat}")
    recs = recommend_jobs(sample_row["Resume"], top_n=5)
    print(recs[["title", "match_score"]].to_string(index=False))


Resume Category: CHEF
                                                                           title  match_score
               Patient Care Technician- 4 Marian- Part time - Day shift- 7am-3pm     0.352307
                                       Population Health Navigator-Boston Region     0.346630
                                  Registered Nurse Care Manager for Jacksonville     0.332148
Staff Nurse, Correctional Health Services **Night Tour** (8pm-8:30am, MWF), GRVC     0.330214
Staff Nurse, Correctional Health Services **Night Tour** (8pm-8:30am, SWS), EMTC     0.324521

Resume Category: ACCOUNTANT
                                                   title  match_score
                                 Senior Staff Accountant     0.516659
Walk-in for GL Accounting (senior Associate) role in EXL     0.498668
                                              Accountant     0.477016
                                              Accountant     0.450298
                                  

In [6]:
keywords_to_check = {
    "CHEF": ["chef", "cook", "culinary", "kitchen"],
    "AVIATION": ["pilot", "aviation", "aircraft", "flight"]
}

for category, keywords in keywords_to_check.items():
    pattern = "|".join(keywords)
    matches = job_corpus[job_corpus["title"].str.lower().str.contains(pattern, na=False)]
    print(f"{category}: {len(matches)} matching job titles found in corpus")
    if len(matches) > 0:
        print(matches["title"].head(5).tolist())
    print()

CHEF: 143 matching job titles found in corpus
['Line Cook - Mercantile Dining & Provision - DEN Airport', 'Cook', 'Executive Chef', '4* Renaissance Marriott Seeks a Talented Chef de Partie!', 'High-End East Bank Club Seeks Talented Demi Chef!']

AVIATION: 46 matching job titles found in corpus
['Delivery Robot Pilot', 'Flight Booking Travel Associate (USA Process)', 'Aviation IT Solutions Business Process Architect', 'Flight control system', 'Service Delivery Manager (Aviation)']



In [7]:
# Test 1: does the recommender work at all for a clean, obvious chef query?
test_recs = recommend_jobs("experienced chef cook culinary kitchen restaurant executive chef", top_n=5)
print("Clean chef query results:")
print(test_recs[["title", "match_score"]].to_string(index=False))

Clean chef query results:
                                                               title  match_score
                             Required Sous Chef for Chain For Kuwait     0.582365
                                                      Executive Chef     0.520052
                                                    Junior Sous Chef     0.517717
                                                           Sous Chef     0.497720
Opening for Kitchen Department in Restaurant Chain_hyderabad/chennai     0.497303


In [8]:
chef_resume = resumes[resumes["Category"] == "CHEF"].iloc[0]
print("Raw resume text (first 500 chars):")
print(chef_resume["Resume"][:500])
print("\n\nDoes it contain the word 'chef'?", "chef" in chef_resume["Resume"].lower())
print("Does it contain 'cook'?", "cook" in chef_resume["Resume"].lower())
print("Does it contain 'culinary'?", "culinary" in chef_resume["Resume"].lower())

Raw resume text (first 500 chars):
         CHEF       Career Focus     I am a nursing student who has recently obtained my CNA license in this state.  I worked as a GNA in the UK and it has been a passion ever since.  I am confident that I would make a wonderful candidate for this position.  From  he beginning of taking my prerequisite classes for Nursing School.  I have ebb driven yet still personable.  My record shows me to muti-task oriented.   I have the experience of always having with and caring deeply for people.  While m


Does it contain the word 'chef'? True
Does it contain 'cook'? True
Does it contain 'culinary'? True


In [9]:
# Check how many resumes might have this label-content mismatch, using Model A's own predictions
resumes["clean_text_check"] = resumes["Resume"].apply(clean_text)
predicted = rf_clf.predict(job_tfidf.transform(resumes["clean_text_check"])) if False else None
# (skip - different vectorizer, just a quick manual spot check instead)

mismatch_count = 0
for cat in resumes["Category"].unique():
    sample = resumes[resumes["Category"] == cat].iloc[0]
    cat_word = cat.lower().split("-")[0]
    if cat_word not in sample["Resume"].lower():
        mismatch_count += 1
        print(f"Possible mismatch - Category: {cat}")

print(f"\n{mismatch_count} out of {resumes['Category'].nunique()} sampled categories show possible label/content mismatch")


0 out of 25 sampled categories show possible label/content mismatch


In [10]:
import pickle

with open("../models/job_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(job_tfidf, f)

with open("../data/processed/job_vectors.pkl", "wb") as f:
    pickle.dump(job_vectors, f)

job_corpus.to_csv("../data/processed/job_corpus_final.csv", index=False)

print("Saved recommender artifacts.")

Saved recommender artifacts.
